# 08 Roboflowデータセットでface検出を転移学習

RoboflowからYOLOv8形式のデータセットを取得し、`yolov8n.pt` を初期重みとして、顔 `face` だけを検出するモデルを学習します。

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")
except Exception:
    ROOT_PATH = Path.cwd()

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

print("ROOT_PATH:", ROOT_PATH)

## Roboflowからデータセットを取得する

このハンズオンではRoboflowの `itsas-workspace/cnn-hands-on` プロジェクトを使います。APIキーだけ自分の値に置き換えてください。

データセットは総枚数200枚をすべて `train` に置く想定です。検証用データは作らず、まずは学習と推論の流れを完成させます。


In [ ]:
from roboflow import Roboflow
import yaml

ROBOFLOW_API_KEY = "xxx"  # 自分のRoboflow APIキーに置き換える

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("itsas-workspace").project("cnn-hands-on")
dataset = project.version(1).download("yolov8")

DATA_YAML = Path(dataset.location) / "data.yaml"

# 200枚すべてをtrainにする運用に合わせて、data.yamlもtrain中心にそろえる。
# val/testフォルダが無い場合でもUltralyticsが読めるように、val/testはtrainを参照させる。
def split_or_train(value):
    if not value:
        return "train/images"
    split_path = Path(value)
    if not split_path.is_absolute():
        split_path = Path(dataset.location) / split_path
    return value if split_path.exists() else "train/images"

data = yaml.safe_load(DATA_YAML.read_text())
data["path"] = str(Path(dataset.location))
data["train"] = "train/images"
data["val"] = split_or_train(data.get("val"))
data["test"] = split_or_train(data.get("test"))
DATA_YAML.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))

print("dataset:", dataset.location)
print("data.yaml:", DATA_YAML)
print(DATA_YAML.read_text())


## yolov8n.ptを初期重みにして学習する

まずは軽い `yolov8n.pt` で流れを確認します。ColabのGPUを有効にしてから実行してください。

全画像をtrainにするため、表示されるvalidation指標は厳密な汎化性能ではなく、学習が進んでいるかを見るための参考値です。


In [ ]:
import torch
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    project="runs/face",
    name="yolov8n_transfer",
    exist_ok=True,
)


## 学習結果を確認する

In [ ]:
from IPython.display import Image, display

run_dir = Path(results.save_dir)
best_model_path = run_dir / "weights" / "best.pt"

print("run_dir:", run_dir)
print("best model:", best_model_path)

display(Image(filename=str(run_dir / "results.png")))


In [ ]:
best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(DATA_YAML), imgsz=640, split="train")
metrics


## 学習画像で推論する

今回は全画像をtrainに置くため、`train/images` から数枚を選んで推論結果を確認します。


In [ ]:
from PIL import Image as PILImage

image_dir = Path(dataset.location) / "train" / "images"
sample_images = sorted(image_dir.glob("*"))[:5]

predict_results = best_model.predict(
    source=[str(path) for path in sample_images],
    conf=0.25,
)

for result in predict_results:
    display(PILImage.fromarray(result.plot()[..., ::-1]))


## Webカメラで撮影してfaceを検出する

In [ ]:
from utils.camera import take_photo

photo_path = take_photo("face_camera.jpg")
display(Image(filename=photo_path))

In [ ]:
from PIL import Image as PILImage

camera_results = best_model.predict(
    source="face_camera.jpg",
    conf=0.25,
)

display(PILImage.fromarray(camera_results[0].plot()[..., ::-1]))
